# TODO
- Read annotation file to dataframe
- check amount of events per label
- make a selection of 1000 events from this set 
- for each event plot the spectrogram and only save those that make sense
- calculate masks for each of these events 
- 

In [ ]:
import pandas as pd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy.signal import spectrogram
import pathlib

%matplotlib widget

In [ ]:
def moc_path(path):
    if not isinstance(path, str):
        return ""
    elif path.begins_with("D:\data"):
        return path.replace("D:\data", "//mnt/fscompute_shared").replace("\\", "/")
    return path.replace(
        "\\\\qarchive\\data_sensors", "//mnt/qarchive_data_sensors"
    ).replace("\\", "/")


def plot_row_event(row):
    data, fs = sf.read(row["Path"])
    fig, ax = plt.subplots()
    start_sample = row["Beg File Samp (samples)"]
    end_sample = int(row["End File Samp (samples)"])
    __, __, Spect = spectrogram(
        data[start_sample:end_sample], fs, nperseg=256, noverlap=128
    )

    ax.pcolormesh(np.log10(np.abs(Spect)))
    plt.show()

# Step 1: Get information on the labeled set

In [ ]:
annotation_location = "/mnt/fs_shared/onderzoek/6. Marine Observation Center/Projects/SoundLib_VLIZ2024/sound_db/sound_bpns/selection_tables/all_annotations.csv"
df = pd.read_csv(annotation_location)
df["Path"] = df["Begin Path"].apply(moc_path)

In [ ]:
balance = np.ceil(
    df["Label"].value_counts() / df["Label"].value_counts().sum() * 1000
).astype(int)
print(balance)

# Step 2: plot the spectrogram of one event 

In [ ]:
row = df.iloc[0]
plot_row_event(row)
# data, fs = sf.read(row['Path'])
# fig, ax = plt.subplots()
# start_sample = row['Beg File Samp (samples)']
# end_sample = int(row['End File Samp (samples)'])
# __,__, Spect = spectrogram(data[start_sample: end_sample], fs, nperseg=256, noverlap=128)

# ax.pcolormesh(np.log10(np.abs(Spect)))

In [ ]:
# take one row and plot the spectrogram of the sound file for the time the event is active
row = df.iloc[0]
print(row)
print(row["Label"])
print(row["Begin Path"])
print(row["Begin Time (s)"])
print(row["End Time (s)"])

In [ ]:
stringpath = "\\\\qarchive\\data_sensors\\rtsys\\PhD_Clea\\Buitenratel_20220119\\records\\220126\\channelA_2022-01-26_22-09-52.wav"
stringpath = stringpath.replace(
    "\\\\qarchive\\data_sensors", "//mnt/qarchive_data_sensors"
)
stringpath = stringpath.replace("\\", "/")
print(stringpath)

In [ ]:
def filter(x):
    if not isinstance(x, str):
        return False
    if x.startswith("D:\\"):
        return True
    return False


filtering = df["Begin Path"].apply(filter)

In [ ]:
df[filtering]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from dearpygui import dearpygui as dpg


# Function to display a plot for a given row
def plot_row(row):
    plt.figure()
    plt.title(f"Row Index: {row.Label}")
    data, fs = sf.read(row["Path"])
    start_sample = row["Beg File Samp (samples)"]
    end_sample = int(row["End File Samp (samples)"])
    __, __, Spect = spectrogram(
        data[start_sample:end_sample], fs, nperseg=256, noverlap=128
    )
    plt.imshow(np.log10(np.abs(Spect)))
    plt.show()


def interactive_row_filter(df):
    """
    Loops through rows of a dataframe, displays a plot for each row, and
    allows the user to decide whether to keep the row or skip it.

    Args:
        df (pd.DataFrame): The input dataframe.

    Returns:
        pd.DataFrame: A new dataframe containing only the rows the user chose to keep.
    """
    selected_rows = []

    for idx, row in df.iterrows():
        # Display the plot for the current row
        plot_row(row)

        # Ask the user if they want to keep the row
        while True:
            user_input = input(f"Keep row {idx}? (y/n): ").strip().lower()
            if user_input in ("y", "n"):
                break
            else:
                print("Invalid input. Please enter 'y' to keep or 'n' to skip.")

        # Append the row index if the user chooses to keep it
        if user_input == "y":
            selected_rows.append(idx)

    # Create a new dataframe with only the selected rows
    return df.loc[selected_rows]


# Example usage (assuming 'your_dataframe' is a predefined DataFrame)
# filtered_df = interactive_row_filter_gui(your_dataframe)
# print(filtered_df)

In [ ]:
filtered_df = interactive_row_filter(df)

In [ ]:
folder = pathlib.Path("/data/bram.cuyx/Gitlab/uw-sim/chosen_rows")
all_anns = pd.DataFrame()
for f in folder.glob("*annotations*.pkl"):
    pd_df = pd.read_pickle(f)
    all_anns = pd.concat([all_anns, pd_df], ignore_index=True)

all_anns[["Label", "Source", "Classification"]]

In [ ]:
folder = pathlib.Path("/data/bram.cuyx/Gitlab/uw-sim/chosen_rows")
all_back = pd.DataFrame()
for f in folder.glob("*background*.pkl"):
    pd_df = pd.read_pickle(f)
    all_back = pd.concat([all_back, pd_df], ignore_index=True)

all_back

In [ ]:
pd.read_pickle(
    "/data/bram.cuyx/Gitlab/uw-sim/chosen_rows/filtered_background_20250122_174903.pkl"
)

In [ ]:
pd.read_csv(
    "/mnt/fs_shared/onderzoek/6. Marine Observation Center/Projects/SoundLib_VLIZ2024/sound_db/sound_bpns/selection_tables/all_annotations.csv"
)

In [ ]:
selection_table = (
    "/mnt/fscompute_shared/sound_classification/total_selections_linux.pkl"
)
select_df = pd.read_pickle(
    selection_table,
)
select_df.columns

In [ ]:
select_df["Begin Time (s)"]

In [ ]:
sel = "/mnt/fs_shared/onderzoek/6. Marine Observation Center/Projects/SoundLib_VLIZ2024/sound_db/sound_bpns/selection_tables/Selection___qarchive_data_sensors_rtsys_PhD_Clea_Grafton_20221027_records_221107.txt"
table = pd.read_table(sel)

In [ ]:
table.columns

In [ ]:
1668981 / 48000